# Processamento de Dados Brutos — Pickle → CSV
### TCC — Análise Preditiva da Popularidade de Jogos na Steam Utilizando Redes Neurais

---

**Discente:** Davi Augusto Farinela da Silva  
**Instituição:** Universidade Federal do Rio Grande do Sul (UFRGS)

---

## Contexto

A coleta de dados da plataforma Steam foi realizada por meio de dois endpoints da API oficial:

1. **`appdetails`** (`store.steampowered.com/api/appdetails`) — retorna os metadados gerais de cada aplicativo (nome, gêneros, preço, plataformas, conquistas, data de lançamento, entre outros).
2. **`appreviews`** (`store.steampowered.com/appreviews/{appid}`) — retorna as métricas detalhadas de avaliação dos jogadores (total de avaliações, avaliações positivas, negativas e a classificação textual oficial da Steam).

Os dados foram salvos progressivamente em **arquivos Pickle** (`.p`) durante a coleta, com salvamento atômico para prevenir corrupção em caso de interrupção. O resultado final é um dicionário Python com **171.769 aplicativos**.

Este notebook documenta a transformação desse dicionário bruto em um **DataFrame tabular** e sua exportação como arquivo CSV, que será utilizado nas etapas posteriores de análise exploratória e modelagem.

---

## Estrutura do Processo

```
Arquivo Pickle bruto (checkpoints_v2/apps_dict-v2.p)
           │
           ▼
  Carregamento em memória (dicionário Python aninhado)
           │
           ▼
  Inspeção da estrutura bruta (verificação de integridade)
           │
           ▼
  Extração e achatamento (flatten) de cada aplicativo → linha do DataFrame
           │
           ▼
  Curadoria: imputação de valores ausentes + remoção de registros inválidos
           │
           ▼
  Exportação como CSV limpo (steam-jogos-v2.csv)
```


---
## 1. Carregamento do Arquivo Pickle

O módulo `pickle` da biblioteca padrão do Python serializa objetos Python em formato binário.
Para carregar o arquivo, utiliza-se o modo de leitura binária (`"rb"`).

O arquivo `apps_dict-v2.p` é um dicionário onde:
- **Chave**: o `appid` numérico do aplicativo na Steam
- **Valor**: um dicionário aninhado com todos os campos retornados pela API


In [ ]:
import pickle
from pathlib import Path

CHECKPOINT_FILE = Path("checkpoints_v2/apps_dict-v2.p")

print(f"Carregando: {CHECKPOINT_FILE.name}")
print("(Arquivo de ~2 GB — pode levar alguns minutos)")

with open(CHECKPOINT_FILE, "rb") as f:
    app_data = pickle.load(f)

print(f"\n✔ Tipo do objeto carregado : {type(app_data)}")
print(f"  Total de aplicativos     : {len(app_data):,}")

---
## 2. Inspeção da Estrutura Bruta

Antes de transformar os dados, é fundamental compreender a estrutura do dicionário bruto retornado pela API.
Os dados são **aninhados**: um aplicativo pode conter listas, sub-dicionários e campos ausentes.
Essa irregularidade é o principal motivo pelo qual o formato Pickle não é adequado para análise direta.

A seguir, examinamos dois exemplos concretos: o Counter-Strike (appid 10) e o Dota 2 (appid 570).


In [ ]:
# Inspecionar as chaves disponíveis para um aplicativo específico
EXEMPLO_APPID = 10  # Counter-Strike

dados_exemplo = app_data[EXEMPLO_APPID]

print(f"── Chaves disponíveis para App ID {EXEMPLO_APPID} ──")
print(sorted(dados_exemplo.keys()))


In [ ]:
# Mostrar campos selecionados no formato bruto (como chegam da API)
print("── Campos brutos — App ID 10 (Counter-Strike) ──\n")

print(f"name       : {dados_exemplo.get('name')}")
print(f"type       : {dados_exemplo.get('type')}")
print(f"is_free    : {dados_exemplo.get('is_free')}")
print(f"\ngenres (lista aninhada):")
print(f"  {dados_exemplo.get('genres')}")
print(f"\ncategories (lista aninhada):")
for cat in dados_exemplo.get('categories', [])[:5]:
    print(f"  {cat}")
print(f"  ...")
print(f"\nrelease_date (dicionário aninhado):")
print(f"  {dados_exemplo.get('release_date')}")
print(f"\nprice_overview (dicionário aninhado):")
print(f"  {dados_exemplo.get('price_overview')}")
print(f"\nexact_reviews (dicionário aninhado — coletado pelo api-atomica-v2):")
print(f"  {dados_exemplo.get('exact_reviews')}")


### Por que o Pickle não pode ser usado diretamente para análise?

| Característica | Pickle (dicionário aninhado) | CSV / DataFrame |
|---|---|---|
| Estrutura | Irregular, hierárquica | Plana, uniforme |
| Campos ausentes | Tratados caso a caso | Coluna com `NaN` |
| Compatível com pandas/sklearn/keras | ❌ Não diretamente | ✅ Sim |
| Leitura por ferramentas externas (Excel, R) | ❌ Não | ✅ Sim |
| Tamanho em disco | ~2 GB (binário compactado) | ~50 MB (texto) |

A transformação consiste em **"achatar"** (flatten) cada dicionário aninhado em uma única linha com colunas fixas.


---
## 3. Funções de Extração

Para lidar com a irregularidade dos dados e evitar erros de chave ausente,
definimos funções auxiliares responsáveis por extrair e normalizar cada grupo de campos.


In [ ]:
import re
import pandas as pd
from datetime import datetime

def parse_release_date(entry: dict):
    """
    Extrai e converte a data de lançamento para o tipo datetime.
    A Steam retorna datas em múltiplos formatos (ex: '21 Aug, 2003', 'Aug 2003'),
    por isso testamos diferentes padrões de formatação.
    """
    raw = entry.get("release_date", {}).get("date", "")
    for fmt in ("%d %b, %Y", "%b %d, %Y", "%b %Y", "%Y"):
        try:
            return datetime.strptime(raw, fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT  # Retorna Not a Time se nenhum formato corresponder


def extract_genres(entry: dict) -> set:
    """
    Extrai os gêneros do aplicativo como um conjunto (set).
    A API retorna uma lista de dicionários: [{'id': '1', 'description': 'Action'}, ...]
    """
    return {g.get("description") for g in entry.get("genres", []) if g.get("description")}


def extract_categories(entry: dict) -> set:
    """
    Extrai as categorias de jogabilidade como um conjunto.
    Exemplos: 'Single-player', 'Multi-player', 'Steam Achievements', etc.
    """
    return {c.get("description") for c in entry.get("categories", []) if c.get("description")}


def extract_platforms(entry: dict) -> dict:
    """Retorna um dicionário com suporte booleano por plataforma.""\"
    p = entry.get("platforms", {})
    return {
        "windows": bool(p.get("windows", False)),
        "mac":     bool(p.get("mac",     False)),
        "linux":   bool(p.get("linux",   False)),
    }


def extract_price_brl(entry: dict):
    """
    Extrai o preço em reais (BRL).
    A API retorna o preço em centavos (ex: 2999 = R$ 29,99).
    Jogos gratuitos não possuem o campo 'price_overview'.
    """
    price = entry.get("price_overview") or {}
    if price:
        return price.get("final", 0) / 100
    return None  # Ausente = jogo gratuito ou indisponível


def extract_supported_languages_count(entry: dict) -> int:
    """
    Conta o número de idiomas suportados.
    O campo vem como string HTML; remove-se as tags e conta-se as vírgulas.
    """
    langs = entry.get("supported_languages", "")
    if not langs:
        return 0
    clean = re.sub(r"<[^>]+>", "", langs)
    return len([l for l in clean.split(",") if l.strip()])


print("✔ Funções de extração definidas com sucesso.")


---
## 4. Extração e Achatamento (Flatten)

Iteramos sobre todos os aplicativos do dicionário e aplicamos as funções de extração
para construir uma lista de dicionários planos — cada um representando uma linha do futuro DataFrame.

Os **gêneros** e **categorias** são transformados em colunas binárias (*one-hot encoding* manual),
onde o valor `True` indica que o aplicativo pertence àquela categoria.


In [ ]:
import numpy as np

# Gêneros que serão transformados em colunas binárias
TARGET_GENRES = [
    "Action", "Adventure", "Casual", "Indie",
    "Massively Multiplayer", "Racing", "RPG",
    "Simulation", "Sports", "Strategy", "Early Access",
]

# Categorias de jogabilidade que serão transformadas em colunas binárias
TARGET_CATEGORIES = [
    "Single-player", "Multi-player", "Co-op", "Online Co-op",
    "Shared/Split Screen Co-op", "PvP", "Online PvP",
    "Steam Achievements", "Steam Cloud", "Steam Trading Cards",
    "Steam Workshop", "Full controller support",
    "Partial Controller Support", "VR Support",
    "Family Sharing", "Valve Anti-Cheat enabled",
]

print("Construindo DataFrame — iterando sobre os aplicativos...")
rows = []

for app_id, d in app_data.items():

    # ── Extração dos campos aninhados ──────────────────────────────────────────
    genres     = extract_genres(d)
    categories = extract_categories(d)
    platforms  = extract_platforms(d)

    # Bloco de avaliações detalhadas (coletado pelo api-atomica-v2.py)
    er = d.get("exact_reviews") or {}

    # ── Construção da linha (dicionário plano) ─────────────────────────────────
    row = {
        # Identificação
        "appid"                    : app_id,
        "name"                     : d.get("name"),
        "type"                     : d.get("type"),

        # Preço e monetização
        "is_free"                  : bool(d.get("is_free", False)),
        "price_brl"                : extract_price_brl(d),

        # Avaliações — obtidas do endpoint appreviews (api-atomica-v2)
        "reviews_total"            : er.get("total_reviews"),
        "reviews_positive"         : er.get("total_positive"),
        "reviews_negative"         : er.get("total_negative"),
        "review_score"             : er.get("review_score"),
        "review_score_desc"        : er.get("review_score_desc"),

        # Conquistas
        "achievements_total"       : (d.get("achievements") or {}).get("total"),

        # Metadados do jogo
        "required_age"             : pd.to_numeric(d.get("required_age", 0), errors="coerce"),
        "metacritic_score"         : pd.to_numeric(
                                        (d.get("metacritic") or {}).get("score"),
                                        errors="coerce"),
        "supported_languages_count": extract_supported_languages_count(d),
        "dlc_count"                : len(d.get("dlc", []) or []),
        "has_demos"                : bool(d.get("demos")),

        # Plataformas (extraídas do sub-dicionário 'platforms')
        "windows"                  : platforms["windows"],
        "mac"                      : platforms["mac"],
        "linux"                    : platforms["linux"],

        # Data de lançamento
        "release_date"             : parse_release_date(d),
    }

    # ── One-hot encoding de gêneros ────────────────────────────────────────────
    for genre in TARGET_GENRES:
        col = f"genre_{genre.lower().replace(' ', '_')}"
        row[col] = genre in genres

    # ── One-hot encoding de categorias ────────────────────────────────────────
    for cat in TARGET_CATEGORIES:
        col = f"cat_{cat.lower().replace(' ', '_').replace('/', '_')}"
        row[col] = cat in categories

    rows.append(row)

# Criar DataFrame
df = pd.DataFrame(rows)

# Derivar colunas de ano e mês a partir da data de lançamento
df["release_year"]  = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month

print(f"\n✔ DataFrame criado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\n── Primeiras 3 linhas (colunas essenciais) ──")
df[["appid", "name", "type", "is_free", "price_brl",
    "reviews_total", "reviews_positive", "review_score_desc", "release_year"]].head(3)


---
## 5. Curadoria dos Dados

A curadoria aplica **regras de negócio** derivadas do comportamento da API da Steam
para eliminar registros inválidos e tratar valores ausentes de forma semanticamente correta.

> **Princípio fundamental:** dados ausentes não significam necessariamente "informação desconhecida".
> Na API da Steam, a ausência de certos campos tem um significado explícito que deve ser preservado.


In [ ]:
print(f"Total antes da curadoria: {len(df):,} aplicativos\n")

# ── Regra 1: Remover aplicativos sem data de lançamento ───────────────────────
# Sem data, não é possível determinar se o app está disponível.
df = df.dropna(subset=["release_date"])
print(f"  Após remover sem data de lançamento :  {len(df):,} apps")

# ── Regra 2: Remover aplicativos com data futura ──────────────────────────────
# Apps não lançados não possuem avaliações, o que distorceria a variável-alvo.
df = df[df["release_date"] <= pd.Timestamp.now()]
print(f"  Após remover não lançados (futuros) :  {len(df):,} apps")

# ── Regra 3: Imputar reviews com 0 ────────────────────────────────────────────
# A ausência do bloco 'exact_reviews' na API indica que o jogo possui
# avaliações insuficientes para compor uma nota — equivalente a 0.
# Remover esses registros introduziria viés de seleção (survivorship bias).
n_reviews_missing = df["reviews_total"].isna().sum()
df["reviews_total"]    = df["reviews_total"].fillna(0)
df["reviews_positive"] = df["reviews_positive"].fillna(0)
df["reviews_negative"] = df["reviews_negative"].fillna(0)
print(f"  Reviews imputadas com 0             :  {n_reviews_missing:,} apps")

# ── Regra 4: Imputar conquistas com 0 ─────────────────────────────────────────
# Jogos sem o campo 'achievements' simplesmente não possuem sistema de conquistas.
n_ach_missing = df["achievements_total"].isna().sum()
df["achievements_total"] = df["achievements_total"].fillna(0)
print(f"  Conquistas imputadas com 0          :  {n_ach_missing:,} apps")

# ── Regra 5: Tratamento de preço ──────────────────────────────────────────────
# 5a. Jogos marcados como gratuitos recebem preço 0.0 (a API não retorna price_overview)
df.loc[df["is_free"], "price_brl"] = 0.0
# 5b. Jogos pagos sem preço listado estão indisponíveis na loja → remover
n_price_missing = df[~df["is_free"] & df["price_brl"].isna()].shape[0]
df = df.dropna(subset=["price_brl"])
print(f"  Jogos pagos sem preço removidos     :  {n_price_missing:,} apps")

# ── Regra 6: Imputar idade mínima com 0 ──────────────────────────────────────
# Sem classificação etária = sem restrição de idade.
df["required_age"] = df["required_age"].fillna(0)

# ── Regra 7: Padronizar review_score_desc ────────────────────────────────────
# A API retorna strings como "1 user reviews", "2 user reviews" para jogos
# com poucas avaliações. Essas strings são mapeadas para "No user reviews".
CANONICAL_SCORES = {
    "Overwhelmingly Positive", "Very Positive", "Mostly Positive", "Positive",
    "Mixed", "Mostly Negative", "Negative", "Overwhelmingly Negative",
}
df["review_score_desc"] = df["review_score_desc"].apply(
    lambda v: v if (pd.notna(v) and v in CANONICAL_SCORES) else "No user reviews"
)

print(f"\n✔ Total após curadoria: {len(df):,} aplicativos")


### Resumo das Regras de Curadoria

| # | Regra | Justificativa |
|---|---|---|
| 1 | Remover sem data de lançamento | Sem data, não é possível verificar se o app foi lançado |
| 2 | Remover com data futura | Apps não lançados não possuem avaliações reais |
| 3 | Imputar `reviews_total = 0` | Ausência do bloco na API = zero avaliações registradas |
| 4 | Imputar `achievements_total = 0` | Ausência do campo = jogo sem sistema de conquistas |
| 5 | `price_brl = 0` para `is_free` | Jogos gratuitos não possuem `price_overview` na API |
| 5 | Remover pagos sem preço | Indica indisponibilidade na loja (fora de catálogo) |
| 6 | Imputar `required_age = 0` | Ausência de classificação etária = sem restrição |
| 7 | Padronizar `review_score_desc` | Strings como "1 user reviews" → "No user reviews" |


---
## 6. Verificação Final de Integridade

Antes da exportação, verificamos se as colunas críticas para a modelagem estão completamente preenchidas.


In [ ]:
cols_check = ["reviews_total", "reviews_positive", "reviews_negative",
              "achievements_total", "price_brl", "required_age", "review_score_desc"]

missing = df[cols_check].isnull().sum()
pct     = (missing / len(df) * 100).round(2)

resultado = pd.DataFrame({"Faltantes": missing, "% do Total": pct})
print("── Dados faltantes nas colunas críticas ──")
print(resultado.to_string())

if resultado["Faltantes"].sum() == 0:
    print("\n✅ Nenhum dado faltante nas colunas críticas. Dataset pronto para exportação.")
else:
    print("\n⚠️ Ainda há dados faltantes. Revisar as regras de curadoria.")


In [ ]:
# Filtrar apenas jogos (type == 'game') para o dataset de modelagem
games = df[df["type"] == "game"].copy()

print(f"── Resumo do Dataset Final ──")
print(f"  Total de apps (todos os tipos) : {len(df):,}")
print(f"  Total de jogos (type=game)     : {len(games):,}")
print(f"  Número de colunas              : {games.shape[1]}")
print(f"  Jogos gratuitos                : {games['is_free'].sum():,} ({games['is_free'].mean()*100:.1f}%)")
print(f"  Jogos pagos                    : {(~games['is_free']).sum():,} ({(~games['is_free']).mean()*100:.1f}%)")
print(f"  Jogos com 0 avaliações         : {(games['reviews_total']==0).sum():,} ({(games['reviews_total']==0).mean()*100:.1f}%)")


---
## 7. Exportação para CSV

O DataFrame final é exportado em dois arquivos:

- **`steam-dados-v2.csv`** — todos os aplicativos processados (jogos, DLCs, trilhas sonoras, etc.)
- **`steam-jogos-v2.csv`** — apenas os aplicativos do tipo `game` (dataset utilizado na modelagem)

A codificação `utf-8-sig` garante compatibilidade com o Excel no Windows (BOM signature).
O argumento `index=False` evita que o índice numérico do DataFrame seja salvo como coluna adicional.


In [ ]:
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

# Dataset completo (todos os tipos de app)
csv_all = OUTPUT_DIR / "steam-dados-v2.csv"
df.to_csv(csv_all, index=False, encoding="utf-8-sig")
print(f"✔ Dataset completo salvo em  : {csv_all}")
print(f"  {len(df):,} apps × {df.shape[1]} colunas")

# Dataset de modelagem (apenas jogos)
csv_games = OUTPUT_DIR / "steam-jogos-v2.csv"
games.to_csv(csv_games, index=False, encoding="utf-8-sig")
print(f"\n✔ Dataset de jogos salvo em  : {csv_games}")
print(f"  {len(games):,} jogos × {games.shape[1]} colunas")

---
## 8. Como Reler o CSV nas Etapas Seguintes

Uma vez exportado, o CSV pode ser carregado em qualquer etapa posterior com uma única linha:

```python
import pandas as pd

df = pd.read_csv("steam-jogos-v2.csv", low_memory=False)
```

O argumento `low_memory=False` é recomendado para datasets com muitas colunas de tipos mistos,
pois força o pandas a inferir os tipos de dados com maior precisão.

---

## Resumo do Pipeline de Dados

```
api-atomica.py      → Coleta V1 (appdetails)
                              ↓
api-atomica-v2.py   → Coleta V2 (appdetails + appreviews) → checkpoints_v2/apps_dict-v2.p
                              ↓
steam-pickle-csv.ipynb → Transformação Pickle → CSV limpo (este notebook)
                              ↓
steam-eda-v2.ipynb  → Análise Exploratória de Dados
                              ↓
[Próximo] Pré-processamento + Treinamento da Rede Neural
```

*Notebook gerado para documentação metodológica do TCC.*
